In [23]:
%reload_ext autoreload
%autoreload 2

In [24]:
from datetime import datetime, timedelta, timezone

import polars as pl

from backtester import io, samplers
from backtester.types import SpotInstrument, OptionInstrument

## Sample data

In [25]:
t0 = datetime(2025, 1, 1, tzinfo=timezone.utc)
tf = datetime(2025, 3, 31, tzinfo=timezone.utc)
dt = timedelta(hours=1)

In [26]:
lf_rate = samplers.get_path_rate(t0, tf, dt)
paths_mark = samplers.get_paths_mark(t0, tf, dt, names=["btc"], s0=[100_000], mu=[0.10], sigma=[0.50])
lf_spot = samplers.to_bars_spot(paths_mark, exchanges="drbt", quotes="usd")
lf_option = samplers.to_bars_option(paths_mark, "drbt", "btc", "usd", rules=[samplers._WEEKLY])

In [27]:
lf_rate.collect()

time_start,time_end,rate
"datetime[μs, UTC]","datetime[μs, UTC]",f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,0.05
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,0.050008
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,0.049946
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,0.050078
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,0.050038
…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,0.049305
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,0.049202
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,0.049255


In [28]:
lf_spot.collect()

time_start,time_end,base,px_mark,exchange,quote,px_bid,px_ask
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,str,f64,f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,"""btc""",100303.760788,"""drbt""","""usd""",99300.72318,101306.798396
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,"""btc""",99973.721522,"""drbt""","""usd""",98973.984307,100973.458738
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,"""btc""",99881.023358,"""drbt""","""usd""",98882.213124,100879.833591
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,"""btc""",99975.43029,"""drbt""","""usd""",98975.675988,100975.184593
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,"""btc""",99309.590482,"""drbt""","""usd""",98316.494577,100302.686387
…,…,…,…,…,…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""btc""",87433.19203,"""drbt""","""usd""",86558.86011,88307.523951
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""btc""",87780.658616,"""drbt""","""usd""",86902.85203,88658.465202
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""btc""",86979.264399,"""drbt""","""usd""",86109.471755,87849.057043


In [29]:
lf_option.collect()

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,iv_bid,iv_ask,iv_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64
2025-03-30 23:00:00 UTC,2025-03-31 00:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 22:00:00 UTC,2025-03-30 23:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",0.99,1.01,1.0
…,…,…,…,…,…,…,…,…,…,…,…
2025-01-03 12:00:00 UTC,2025-01-03 13:00:00 UTC,"""drbt""","""btc""","""usd""",85000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",0.99,1.01,1.0
2025-01-03 11:00:00 UTC,2025-01-03 12:00:00 UTC,"""drbt""","""btc""","""usd""",85000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",0.99,1.01,1.0
2025-01-03 10:00:00 UTC,2025-01-03 11:00:00 UTC,"""drbt""","""btc""","""usd""",85000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",0.99,1.01,1.0


## `_get_lf_priced`

In [30]:
lf_priced = io._get_lf_priced(
    lf_rate, lf_spot, lf_option,
    option_exchange="drbt",
    option_base="btc",
    option_quote="usd",
    spot_exchange="drbt",
    spot_base="btc",
    spot_quote="usd",
)
lf_priced.collect().to_native()

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,spot,rate,iv_bid,iv_ask,iv_mark,px_bid,px_ask,px_mark,delta,gamma,vega,theta,rho
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2025-03-30 23:00:00 UTC,2025-03-31 00:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",87673.573177,0.04882,0.99,1.01,1.0,68.85443,78.719543,73.680714,0.021576,0.000005,493.255644,-20860.453481,21.582965
2025-03-30 22:00:00 UTC,2025-03-30 23:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",86748.28913,0.048931,0.99,1.01,1.0,53.832545,62.03302,57.836889,0.017476,0.000005,410.023731,-17172.964437,17.478381
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",86979.264399,0.049255,0.99,1.01,1.0,59.747599,68.631616,64.089009,0.019032,0.000005,444.200837,-18430.97831,19.255697
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",87780.658616,0.049202,0.99,1.01,1.0,78.191972,89.074611,83.520597,0.023801,0.000006,544.131904,-22370.745228,24.499567
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-28 08:00:00 UTC,2025-04-04 08:00:00 UTC,"""c""",87433.19203,0.049305,0.99,1.01,1.0,72.919237,83.260272,77.979893,0.022383,0.000005,517.051746,-21060.113366,23.165852
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-01-03 12:00:00 UTC,2025-01-03 13:00:00 UTC,"""drbt""","""btc""","""usd""",85000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",105645.527404,0.049948,0.99,1.01,1.0,285.098812,313.656217,299.194901,-0.047555,0.000007,1427.870236,-38103.392654,-99.048903
2025-01-03 11:00:00 UTC,2025-01-03 12:00:00 UTC,"""drbt""","""btc""","""usd""",85000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",105994.135251,0.049857,0.99,1.01,1.0,273.512028,301.329366,287.238514,-0.045661,0.000007,1390.8669,-36891.335847,-95.985056
2025-01-03 10:00:00 UTC,2025-01-03 11:00:00 UTC,"""drbt""","""btc""","""usd""",85000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""p""",105948.880838,0.049776,0.99,1.01,1.0,279.6154,307.87385,293.561381,-0.046413,0.000007,1412.922502,-37247.959376,-98.151107


In [31]:
# Sanity checks: prices positive, Greeks finite
df_priced = lf_priced.collect()
assert (df_priced["px_mark"] >= 0).all(), "px_mark should be non-negative"
assert df_priced["delta"].is_finite().all(), "delta should be finite"
assert df_priced["gamma"].is_finite().all(), "gamma should be finite"
assert df_priced["vega"].is_finite().all(), "vega should be finite"
assert df_priced["theta"].is_finite().all(), "theta should be finite"
assert df_priced["rho"].is_finite().all(), "rho should be finite"
print(f"priced rows: {len(df_priced):,}")

priced rows: 20,800


In [32]:
dfp: pl.DataFrame = df_priced.to_native()
dfp.filter(pl.col("px_mark") <= 0).select([
    "time_start", "time_end", "exchange", "base", "quote", "strike", "listing", "expiry", "kind", "px_mark"
])

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,px_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64
2025-01-10 00:00:00 UTC,2025-01-10 01:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
2025-01-10 01:00:00 UTC,2025-01-10 02:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
2025-01-10 02:00:00 UTC,2025-01-10 03:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
2025-01-10 03:00:00 UTC,2025-01-10 04:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
2025-01-10 04:00:00 UTC,2025-01-10 05:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.0
…,…,…,…,…,…,…,…,…,…
2025-03-28 07:00:00 UTC,2025-03-28 08:00:00 UTC,"""drbt""","""btc""","""usd""",110000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""c""",0.0
2025-03-28 06:00:00 UTC,2025-03-28 07:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""c""",0.0
2025-03-28 07:00:00 UTC,2025-03-28 08:00:00 UTC,"""drbt""","""btc""","""usd""",100000.0,2025-03-21 08:00:00 UTC,2025-03-28 08:00:00 UTC,"""c""",0.0


## `get_bars_spot`

In [33]:
spot = SpotInstrument(exchange="drbt", base="btc", quote="usd")

In [34]:
# Full range
io.get_bars_spot(lf_spot, spot).collect()

time_start,time_end,base,px_mark,exchange,quote,px_bid,px_ask
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,str,f64,f64
2025-01-01 00:00:00 UTC,2025-01-01 01:00:00 UTC,"""btc""",100303.760788,"""drbt""","""usd""",99300.72318,101306.798396
2025-01-01 01:00:00 UTC,2025-01-01 02:00:00 UTC,"""btc""",99973.721522,"""drbt""","""usd""",98973.984307,100973.458738
2025-01-01 02:00:00 UTC,2025-01-01 03:00:00 UTC,"""btc""",99881.023358,"""drbt""","""usd""",98882.213124,100879.833591
2025-01-01 03:00:00 UTC,2025-01-01 04:00:00 UTC,"""btc""",99975.43029,"""drbt""","""usd""",98975.675988,100975.184593
2025-01-01 04:00:00 UTC,2025-01-01 05:00:00 UTC,"""btc""",99309.590482,"""drbt""","""usd""",98316.494577,100302.686387
…,…,…,…,…,…,…,…
2025-03-30 19:00:00 UTC,2025-03-30 20:00:00 UTC,"""btc""",87433.19203,"""drbt""","""usd""",86558.86011,88307.523951
2025-03-30 20:00:00 UTC,2025-03-30 21:00:00 UTC,"""btc""",87780.658616,"""drbt""","""usd""",86902.85203,88658.465202
2025-03-30 21:00:00 UTC,2025-03-30 22:00:00 UTC,"""btc""",86979.264399,"""drbt""","""usd""",86109.471755,87849.057043


In [35]:
# With time filter
start = datetime(2025, 2, 1, tzinfo=timezone.utc)
end = datetime(2025, 2, 28, tzinfo=timezone.utc)
df_spot_filtered = io.get_bars_spot(lf_spot, spot, start_time=start, end_time=end).collect()
assert (df_spot_filtered["time_start"] > start).all()
assert (df_spot_filtered["time_end"] < end).all()
df_spot_filtered

time_start,time_end,base,px_mark,exchange,quote,px_bid,px_ask
"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,str,f64,f64
2025-02-01 01:00:00 UTC,2025-02-01 02:00:00 UTC,"""btc""",91353.828257,"""drbt""","""usd""",90440.289974,92267.36654
2025-02-01 02:00:00 UTC,2025-02-01 03:00:00 UTC,"""btc""",90947.509726,"""drbt""","""usd""",90038.034629,91856.984824
2025-02-01 03:00:00 UTC,2025-02-01 04:00:00 UTC,"""btc""",90930.716818,"""drbt""","""usd""",90021.409649,91840.023986
2025-02-01 04:00:00 UTC,2025-02-01 05:00:00 UTC,"""btc""",90903.693734,"""drbt""","""usd""",89994.656797,91812.730672
2025-02-01 05:00:00 UTC,2025-02-01 06:00:00 UTC,"""btc""",91334.337481,"""drbt""","""usd""",90420.994106,92247.680856
…,…,…,…,…,…,…,…
2025-02-27 18:00:00 UTC,2025-02-27 19:00:00 UTC,"""btc""",82407.208768,"""drbt""","""usd""",81583.136681,83231.280856
2025-02-27 19:00:00 UTC,2025-02-27 20:00:00 UTC,"""btc""",82409.801951,"""drbt""","""usd""",81585.703932,83233.899971
2025-02-27 20:00:00 UTC,2025-02-27 21:00:00 UTC,"""btc""",82656.139736,"""drbt""","""usd""",81829.578338,83482.701133


## `get_bars_option`

In [36]:
# Pick an arbitrary option from the sampled data
row = lf_option.select(["exchange", "base", "quote", "strike", "listing", "expiry", "kind"]).head(1).collect()
opt = OptionInstrument(
    exchange=row["exchange"].item(),
    base=row["base"].item(),
    quote=row["quote"].item(),
    strike=row["strike"].item(),
    listing=row["listing"].item(),
    expiry=row["expiry"].item(),
    kind=row["kind"].item(),
)
opt

OptionInstrument(exchange='drbt', base='btc', quote='usd', strike=130000.0, listing=datetime.datetime(2025, 1, 3, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), expiry=datetime.datetime(2025, 1, 10, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), kind='c')

In [37]:
# Full range
io.get_bars_option(lf_option, opt).collect()

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,iv_bid,iv_ask,iv_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64
2025-01-03 08:00:00 UTC,2025-01-03 09:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 09:00:00 UTC,2025-01-03 10:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 10:00:00 UTC,2025-01-03 11:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 11:00:00 UTC,2025-01-03 12:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-03 12:00:00 UTC,2025-01-03 13:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
…,…,…,…,…,…,…,…,…,…,…,…
2025-01-10 03:00:00 UTC,2025-01-10 04:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-10 04:00:00 UTC,2025-01-10 05:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0
2025-01-10 05:00:00 UTC,2025-01-10 06:00:00 UTC,"""drbt""","""btc""","""usd""",130000.0,2025-01-03 08:00:00 UTC,2025-01-10 08:00:00 UTC,"""c""",0.99,1.01,1.0


In [38]:
# With time filter
df_opt_filtered = io.get_bars_option(lf_option, opt, start_time=start, end_time=end).collect()
if len(df_opt_filtered) > 0:
    assert (df_opt_filtered["time_start"] > start).all()
    assert (df_opt_filtered["time_end"] < end).all()
df_opt_filtered

time_start,time_end,exchange,base,quote,strike,listing,expiry,kind,iv_bid,iv_ask,iv_mark
"datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,f64,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,f64,f64


## `get_target_option`

In [39]:
target = io.get_target_option(
    lf_rate, lf_spot, lf_option,
    option_exchange="drbt",
    option_base="btc",
    option_quote="usd",
    option_kind="c",
    spot_instrument=spot,
    target_time=datetime(2025, 2, 1, tzinfo=timezone.utc),
    target_delta=0.5,
    target_tenor=timedelta(days=30),
)
target

OptionInstrument(exchange='drbt', base='btc', quote='usd', strike=92000.0, listing=datetime.datetime(2025, 1, 31, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), expiry=datetime.datetime(2025, 2, 7, 8, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), kind='c')

In [40]:
assert isinstance(target, OptionInstrument)
assert target.kind == "c"
assert target.exchange == "drbt"
assert target.base == "btc"
print(f"strike={target.strike}, listing={target.listing}, expiry={target.expiry}")

strike=92000.0, listing=2025-01-31 08:00:00+00:00, expiry=2025-02-07 08:00:00+00:00
